# Which random process describes read depth along the genome?

[depth_autocorrelation.ipynb](depth_autocorrelation.ipynb) showed that depth in neighbouring 500 bp windows is strongly correlated, and that the correlation dies away over tens of kb.
This notebook fits several candidate **stochastic processes** (random processes that generate a series along the genome) to the same data and asks which one describes it best.

Candidates:

| family | models | idea |
|---|---|---|
| no correlation | iid Gaussian | every window independent (the baseline to beat) |
| autoregressive | AR(1) … AR(20) | each window = weighted sum of the previous *p* windows + noise |
| ARMA | ARMA(1,1), ARMA(2,1), ARMA(2,2) | AR plus a weighted sum of past *noise* terms |
| Gaussian process | exponential, squared exponential, Matérn 3/2, Matérn 5/2, two exponentials | correlation is a chosen function of distance (the *kernel*) |
| hidden Markov model | 2 and 3 states | the genome switches between a few depth levels |

Every model is fitted by **maximum likelihood** to exactly the same numbers for each sample, then compared with **AIC** and **BIC**.

Data: every **sWGA** sample with a `<sample>.window_stats.tsv` in `$PF8/output/`.
Run the cells one at a time, top to bottom (`conda activate krsim`, then `jupyter lab`). Step 12 (fit everything for every sample) takes a few minutes.

## Step 1: paths and settings

In [ ]:
import glob
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from scipy.optimize import minimize
from scipy.signal import lfilter
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import pacf

# Folder for the Pf8 data (change this on another laptop)
PF8 = "/Users/jbansal/Desktop/IMPERIAL/pf8"

WINDOW = 500      # bp per window, as made by pf8_profiles
PSEUDO = 0.05     # added before taking the log, so windows with zero depth do not give -inf (same as notebook 1)
TRIM_KB = 50      # drop this many kb from each end of every chromosome (see Step 3)
K = 200           # how many previous windows the likelihood conditions on (200 windows = 100 kb; see Step 4)
AR_ORDERS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20]
MAX_LAG = 200     # lags for the ACF plots (as in notebook 1)

## Step 2: the sWGA samples that have window stats

In [ ]:
files = sorted(glob.glob(f"{PF8}/output/*.window_stats.tsv"))
info = pd.read_csv(f"{PF8}/input/Pf8_samples.txt", sep="\t", index_col="Sample")

samples = [os.path.basename(f).replace(".window_stats.tsv", "") for f in files]
samples = [s for s in samples if info.loc[s, "Sample type"] == "sWGA"]
info.loc[samples, ["Country", "Year", "Population", "Sample type"]]

## Step 3: the series each model is fitted to

As in notebook 1: `y = log(mean depth / sample median + PSEUDO)` for each 500 bp window of the 14 nuclear chromosomes.

**Chromosome ends.** Most zero-depth windows are in the subtelomeric regions (var genes and repeats, which sWGA primers and short-read mapping both handle badly).
A zero window gives `y = log(0.05) ≈ -3`, far below everything else, and a run of them looks like a huge, long-lasting dip that no stationary process explains well.
`TRIM_KB = 50` drops the first and last 50 kb of each chromosome, which removes most of them while keeping the series unbroken.
Set `TRIM_KB = 0` to keep everything (the comparison is still fair, because every model sees the same windows).

**Mean.** Each model describes how `y` wanders *around its average*, so the sample's average `y` is subtracted first.
That average is one fitted parameter, counted in every model.

**Chromosomes are separate series.** The process restarts at every chromosome, and the log-likelihood of a sample is the sum over its 14 chromosomes.

In [ ]:
def load(sample):
    df = pd.read_csv(f"{PF8}/output/{sample}.window_stats.tsv", sep="\t")
    df = df[~df["chrom"].str.contains("API|MIT")]
    y_all = np.log(df["mean"] / df["mean"].median() + PSEUDO)
    trim = TRIM_KB * 1000 // WINDOW
    ys = [y_all[df["chrom"] == c].to_numpy()[trim:(df["chrom"] == c).sum() - trim] for c in df["chrom"].unique()]
    n_zero = sum((y == np.log(PSEUDO)).sum() for y in ys)
    mu = np.concatenate(ys).mean()
    return [y - mu for y in ys], n_zero

data, n_zero = {}, {}
for s in samples:
    data[s], n_zero[s] = load(s)

pd.DataFrame({
    "windows used": [sum(len(y) for y in data[s]) for s in samples],
    "zero windows left": [n_zero[s] for s in samples],
    "var(y)": [np.concatenate(data[s]).var() for s in samples],
}, index=samples)

## Likelihood, AIC and BIC

**Likelihood.** A model with fixed parameters gives a probability (density) to any series. The likelihood is that probability for *the series we actually observed*.
We work with its log (the **log-likelihood**, LL), because the probability of 44,000 windows is an astronomically small number.
**Maximum likelihood** = choose the parameters that make the observed series most probable. Higher LL = better fit.

**The catch.** Adding parameters can never lower the best LL (AR(2) can always copy AR(1) by setting its second coefficient to 0), so LL alone always favours the biggest model.
AIC and BIC add a penalty for each parameter (k = number of parameters, n = number of windows):

- **AIC** = 2k − 2 LL
- **BIC** = k ln(n) − 2 LL. With n ≈ 44,000, ln(n) ≈ 10.7, so BIC charges each parameter about 5× more than AIC and prefers smaller models.

**Lower AIC / BIC is better.** Only differences matter (ΔAIC = AIC − AIC of the best model). Rough rule: Δ < 2 no real difference, Δ > 10 strong evidence for the better model.
With this much data, differences between models are often in the hundreds or thousands.

## Step 4: one likelihood for every model (Durbin–Levinson)

Every model here except the HMM is a **stationary Gaussian process**: `y` is Gaussian, and the covariance of two windows depends only on the distance between them, `γ(k)` for a lag of k windows.
Such a model is fully described by its **autocovariance** `γ(0), γ(1), γ(2), …` (the ACF is `γ(k) / γ(0)`).

The likelihood can be written window by window: predict each window from the ones before it, and score how far off the prediction was:

LL = Σ_t log Normal( y_t ; prediction from y_1 … y_{t−1}, prediction variance v_t )

The **Durbin–Levinson recursion** turns `γ` into those prediction coefficients and variances, step by step.
Along the way it also gives the **partial autocorrelations** (PACF), which is how AR models are parameterised below.

**Shortcut.** Each window is predicted from at most the previous `K = 200` windows (100 kb) instead of the whole chromosome so far.
The correlation is near 0 by 100 kb (notebook 1), so this changes the LL very little (checked against the exact LL in Step 13). For AR(p) with p ≤ K it is exact.

In [ ]:
def levinson(gamma=None, pacf_values=None, v0=None):
    # Durbin-Levinson recursion, from the autocovariance gamma[0..K]
    # or from the partial autocorrelations a_1..a_K plus the variance v0.
    # Returns phi (row m = coefficients of the best predictor from the previous m windows)
    # and v (v[m] = variance of that prediction's error).
    if gamma is not None:
        n_lags, v0 = len(gamma) - 1, gamma[0]
    else:
        n_lags = len(pacf_values)
    phi = np.zeros((n_lags + 1, n_lags + 1))
    v = np.empty(n_lags + 1)
    v[0] = v0
    for m in range(1, n_lags + 1):
        if gamma is not None:
            a = (gamma[m] - phi[m - 1, 1:m] @ gamma[m - 1:0:-1]) / v[m - 1]
        else:
            a = pacf_values[m - 1]
        phi[m, m] = a
        phi[m, 1:m] = phi[m - 1, 1:m] - a * phi[m - 1, m - 1:0:-1]
        v[m] = v[m - 1] * (1 - a * a)
    return phi, v


def lag_matrix(y):
    # row t = the windows before window t, nearest first (only the first K rows are needed)
    L = np.zeros((K, K))
    for t in range(1, K):
        L[t, :t] = y[t - 1::-1][:t]
    return L


def loglik(ys, Ls, phi, v):
    # Gaussian log-likelihood of the chromosomes ys, each window predicted from up to K previous ones
    if not (np.all(np.isfinite(v)) and np.all(v > 0)):
        return -np.inf
    ll = 0.0
    for y, L in zip(ys, Ls):
        e = lfilter(np.r_[1, -phi[K, 1:]], [1], y)      # prediction errors once K windows are available
        e[:K] = y[:K] - (phi[:K, 1:] * L).sum(1)        # the first K windows have fewer before them
        var = np.r_[v[:K], np.full(len(y) - K, v[K])]
        ll += -0.5 * np.sum(np.log(2 * np.pi * var) + e * e / var)
    return ll

lags = {s: [lag_matrix(y) for y in data[s]] for s in samples}

## Step 5: check the likelihood against statsmodels

For an AR(1) on one chromosome, our LL should match statsmodels' exact ARIMA likelihood at the same parameters.

In [ ]:
y = data[samples[0]][6]                         # chromosome 7 of the first sample
fit = ARIMA(y, order=(1, 0, 0), trend="n").fit()
phi1, sigma2 = fit.params
v0 = sigma2 / (1 - phi1 ** 2)                    # variance of y implied by AR(1)
a = np.zeros(K); a[0] = phi1
ours = loglik([y], [lag_matrix(y)], *levinson(pacf_values=a, v0=v0))
print(f"AR(1) phi = {phi1:.4f}   statsmodels LL = {fit.llf:.3f}   ours = {ours:.3f}")

## Step 6: fitting machinery

Each model below is a pair of functions:

- `build(x)` turns the model's parameters `x` into the Durbin–Levinson output (`phi`, `v`),
- `start(ys)` gives starting values (one or several) for the optimiser.

Parameters are put on a scale where any real number is allowed (logs of variances and length-scales, and `atanh` of partial autocorrelations, which keeps AR models stationary),
and `scipy.optimize.minimize` searches for the maximum likelihood. With several starting values, the best result is kept.

In [ ]:
def fit(ys, Ls, build, starts):
    def neg_ll(x):
        ll = loglik(ys, Ls, *build(x))
        return -ll if np.isfinite(ll) else 1e20
    best = None
    for x0 in starts:
        r = minimize(neg_ll, x0, method="L-BFGS-B", bounds=[(-12, 12)] * len(x0))
        if best is None or r.fun < best.fun:
            best = r
    return best.x, -best.fun

lag_d = np.arange(K + 1)   # lags 0..K, in windows

## Step 7: model 1, iid Gaussian (no correlation)

Every window is drawn independently from Normal(0, σ²). ACF: 1 at lag 0, 0 at every other lag.
Parameters: σ² (+ the mean). It cannot be right (notebook 1 shows strong correlation), but it is the baseline: how much does modelling the correlation improve the fit?

In [ ]:
def iid_model():
    build = lambda x: levinson(pacf_values=np.zeros(K), v0=np.exp(x[0]))
    start = lambda ys: [np.r_[np.log(np.concatenate(ys).var())]]
    return build, start

## Step 8: autoregressive models, AR(p)

**AR(1):** `y_t = φ y_{t−1} + ε_t`, with independent noise `ε_t ~ Normal(0, σ²)`. Each window is pulled towards the previous one.
Its ACF is `φ^k`: an **exponential decay**, a straight line on a log y-axis. The correlation falls by a factor e every `−1/ln φ` windows (the *correlation length*).

**AR(p):** `y_t = φ_1 y_{t−1} + … + φ_p y_{t−p} + ε_t`. More lags let the ACF take more shapes, e.g. a fast drop followed by a slow tail (a mix of several exponentials).
Its PACF is non-zero up to lag p and exactly 0 after, which is why the PACF suggests the order.

Parameters: p partial autocorrelations (each between −1 and 1) and the variance (+ the mean) → p + 2.
The partial autocorrelations of the data are used as the starting values.

In [ ]:
def ar_model(p):
    def build(x):
        a = np.zeros(K)
        a[:p] = np.tanh(x[1:])
        return levinson(pacf_values=a, v0=np.exp(x[0]))
    def start(ys):
        y = np.concatenate(ys)
        return [np.r_[np.log(y.var()), np.arctanh(np.clip(pacf(y, nlags=p)[1:], -0.99, 0.99))]]
    return build, start

## Step 9: ARMA models

**ARMA(p, q)** adds a moving-average part: `y_t = φ_1 y_{t−1} + … + φ_p y_{t−p} + ε_t + θ_1 ε_{t−1} + … + θ_q ε_{t−q}`.
The MA terms let a shock last a few extra windows, which mostly changes the first few lags of the ACF.

Why these are worth testing:

- **AR(1) + independent noise = ARMA(1,1).** Depth in a window = a smoothly wandering level plus independent sampling noise.
- **Two AR(1) processes added together = ARMA(2,1).** A *fast* process (correlation over a few windows) plus a *slow* one (broad regions of high or low amplification). Its ACF is a weighted mix of two exponentials, `w φ_fast^k + (1 − w) φ_slow^k`: a fast drop, then a long tail, the shape seen in notebook 1.
  The reverse is not true: ARMA(2,1) can also make ACF shapes that no sum of two AR(1)s can, e.g. one that starts out *rounded* (flat at lag 0, then falling), so ARMA(2,1) is the more flexible model.
- **Two AR(1) + independent noise = ARMA(2,2).**

The autocovariance comes from writing the ARMA as a sum of past noise terms (the ψ weights), `γ(k) = σ² Σ_j ψ_j ψ_{j+k}`.
Parameters: p + q coefficients and σ² (+ the mean) → p + q + 2.

In [ ]:
def pacf_to_coefs(r):
    # partial autocorrelations in (-1, 1) -> coefficients of a stationary AR polynomial
    c = np.zeros(0)
    for m, a in enumerate(r, 1):
        c = np.r_[c - a * c[::-1], a]
    return c

N_PSI = 20000   # psi weights kept (enough for correlation lengths of thousands of windows)

def arma_gamma(x, p, q, lags):
    sigma2 = np.exp(x[0])
    ar = np.r_[1, -pacf_to_coefs(np.tanh(x[1:1 + p]))]
    ma = np.r_[1, -pacf_to_coefs(np.tanh(x[1 + p:]))]   # keeps the MA part invertible
    impulse = np.zeros(N_PSI); impulse[0] = 1
    psi = lfilter(ma, ar, impulse)
    return sigma2 * np.array([psi[:N_PSI - k] @ psi[k:] for k in lags])

def arma_model(p, q):
    build = lambda x: levinson(gamma=arma_gamma(x, p, q, lag_d))
    def start(ys):
        y = np.concatenate(ys)
        lv = np.log(y.var())
        slow = np.r_[np.arctanh(0.98), np.zeros(p - 1)]
        return [np.r_[lv - 2, slow, np.full(q, np.arctanh(-0.5))],
                np.r_[lv - 2, slow, np.full(q, np.arctanh(0.5))],
                np.r_[lv, np.arctanh(np.clip(pacf(y, nlags=p)[1:], -0.99, 0.99)), np.zeros(q)]]
    return build, start

## Step 10: Gaussian processes

A Gaussian process (GP) says: the covariance of two windows d windows apart is a chosen function of d, the **kernel** `k(d)`.
Here each kernel has a variance σ² and a **length-scale** ℓ (how far the correlation reaches), plus a **nugget** τ², independent noise added to every window (`γ(0) = σ² + τ²`, `γ(d) = k(d)` for d > 0).

| kernel | k(d) | ACF shape near 0 | process looks |
|---|---|---|---|
| exponential | σ² exp(−d/ℓ) | sharp drop (a corner at 0) | rough, jagged |
| Matérn 3/2 | σ² (1 + √3 d/ℓ) exp(−√3 d/ℓ) | flatter at 0 | once smooth |
| Matérn 5/2 | σ² (1 + √5 d/ℓ + 5d²/3ℓ²) exp(−√5 d/ℓ) | flatter still | twice smooth |
| squared exponential | σ² exp(−d²/2ℓ²) | completely flat at 0 | very smooth |
| two exponentials | σ₁² exp(−d/ℓ₁) + σ₂² exp(−d/ℓ₂) | fast drop, then slow tail | rough local + broad regions |

Parameters: 3 (σ², ℓ, τ²) + the mean = 4; two exponentials: 5 + the mean = 6.

**Links to the AR family.** On an evenly spaced grid, the exponential kernel *is* AR(1) with φ = exp(−1/ℓ); exponential + nugget is ARMA(1,1); two exponentials + nugget is a special case of ARMA(2,2).
So GP exponential and AR(1) should give the same LL (and exponential + nugget = ARMA(1,1) too).
Two exponentials only allows positive weights on two decaying exponentials, so ARMA(2,1)/ARMA(2,2) can match it or do better; if they do much better, the data needs a shape that sums of exponentials cannot make.
The Matérn and squared exponential kernels are genuinely different shapes: *rounded* at short distances instead of a sharp drop.

In [ ]:
KERNELS = {
    "exponential": lambda d, l: np.exp(-d / l),
    "Matern 3/2": lambda d, l: (1 + np.sqrt(3) * d / l) * np.exp(-np.sqrt(3) * d / l),
    "Matern 5/2": lambda d, l: (1 + np.sqrt(5) * d / l + 5 * d ** 2 / (3 * l ** 2)) * np.exp(-np.sqrt(5) * d / l),
    "squared exp": lambda d, l: np.exp(-d ** 2 / (2 * l ** 2)),
}

def gp_gamma(x, kernel, lags):
    # x = log sigma2, log length-scale (windows), log nugget
    s2, l, t2 = np.exp(x)
    g = s2 * KERNELS[kernel](lags, l)
    g[lags == 0] += t2
    return g

def gp2_gamma(x, lags):
    # two exponentials: x = log s1, log l1, log s2, log l2, log nugget
    s1, l1, s2, l2, t2 = np.exp(x)
    g = s1 * np.exp(-lags / l1) + s2 * np.exp(-lags / l2)
    g[lags == 0] += t2
    return g

def gp_model(kernel):
    build = lambda x: levinson(gamma=gp_gamma(x, kernel, lag_d))
    def start(ys):
        lv = np.log(np.concatenate(ys).var())
        return [np.r_[lv, np.log(l), lv - 3] for l in (2, 10, 40)]
    return build, start

def gp2_model():
    build = lambda x: levinson(gamma=gp2_gamma(x, lag_d))
    def start(ys):
        lv = np.log(np.concatenate(ys).var())
        return [np.r_[lv - 0.7, np.log(l1), lv - 0.7, np.log(l2), lv - 3] for l1, l2 in ((1, 20), (3, 60), (5, 150))]
    return build, start

# what the kernels look like (length-scale 10 windows = 5 kb)
d = np.arange(0, 61)
fig, ax = plt.subplots(figsize=(8, 4))
for name, k in KERNELS.items():
    ax.plot(d * WINDOW / 1e3, k(d, 10), label=name)
ax.plot(d * WINDOW / 1e3, 0.6 * np.exp(-d / 2) + 0.4 * np.exp(-d / 40), label="two exponentials (2 and 40 windows)")
ax.set_xlabel("distance (kb)")
ax.set_ylabel("correlation")
ax.legend()
fig.tight_layout()

## Step 11: hidden Markov models

A different idea: the genome is in one of a few hidden **states** (e.g. "well amplified", "poorly amplified"), each with its own mean and variance of `y`.
Moving along the genome, the state stays the same or switches with fixed probabilities. Windows are independent *given* the state, so all the correlation comes from long runs of the same state.
The ACF of a 2-state HMM is again an exponential decay, plus independent noise.

Parameters for s states: s means, s variances, s(s − 1) switching probabilities, s − 1 starting probabilities (+ the mean subtracted in Step 3, counted for every model).
Fitted with `hmmlearn` (the EM algorithm, from several random starts). This is not a Gaussian process, so it has its own likelihood.

In [ ]:
def fit_hmm(ys, n_states, seeds=(0, 1, 2)):
    Y = np.concatenate(ys)[:, None]
    lengths = [len(y) for y in ys]
    best, best_ll = None, -np.inf
    for seed in seeds:
        m = GaussianHMM(n_components=n_states, covariance_type="diag", n_iter=300, random_state=seed)
        m.fit(Y, lengths)
        ll = m.score(Y, lengths)
        if ll > best_ll:
            best, best_ll = m, ll
    return best, best_ll

hmm_params = lambda s: s + s + s * (s - 1) + (s - 1)

## Step 12: fit every model to every sample

The list of models, each with its number of parameters (not counting the mean, which is added below).
This cell prints one line per sample; it takes a few minutes in total.

In [ ]:
MODELS = {"iid": (iid_model(), 1)}
for p in AR_ORDERS:
    MODELS[f"AR({p})"] = (ar_model(p), p + 1)
for p, q in [(1, 1), (2, 1), (2, 2)]:
    MODELS[f"ARMA({p},{q})"] = (arma_model(p, q), p + q + 1)
for kernel in KERNELS:
    MODELS[f"GP {kernel}"] = (gp_model(kernel), 3)
MODELS["GP two exponentials"] = (gp2_model(), 5)

rows, fitted, hmms = [], {}, {}
for s in samples:
    t0 = time.time()
    ys, Ls = data[s], lags[s]
    n = sum(len(y) for y in ys)
    for name, ((build, start), k) in MODELS.items():
        x, ll = fit(ys, Ls, build, start(ys))
        fitted[s, name] = x
        rows.append({"sample": s, "model": name, "LL": ll, "k": k + 1, "n": n})
    for n_states in (2, 3):
        m, ll = fit_hmm(ys, n_states)
        hmms[s, n_states] = m
        rows.append({"sample": s, "model": f"HMM {n_states} states", "LL": ll, "k": hmm_params(n_states) + 1, "n": n})
    print(f"{s}: {time.time() - t0:.0f} s")

res = pd.DataFrame(rows)
res["AIC"] = 2 * res["k"] - 2 * res["LL"]
res["BIC"] = res["k"] * np.log(res["n"]) - 2 * res["LL"]
res["dAIC"] = res["AIC"] - res.groupby("sample")["AIC"].transform("min")
res["dBIC"] = res["BIC"] - res.groupby("sample")["BIC"].transform("min")

## Step 13: is the 100 kb shortcut safe?

Recompute the LL of every Gaussian-process and ARMA fit for one sample *exactly* (each window predicted from the whole chromosome before it) and compare with Step 12.

In [ ]:
def exact_loglik(ys, gamma):
    # Durbin-Levinson over the full chromosome length, one step at a time (memory stays small)
    n_max = max(len(y) for y in ys)
    e2v = [np.zeros(len(y)) for y in ys]
    logv = [np.zeros(len(y)) for y in ys]
    phi, v = np.zeros(0), gamma[0]
    for m in range(n_max):
        for i, y in enumerate(ys):
            if m < len(y):
                e = y[m] - phi @ y[m - 1::-1][:m] if m else y[0]
                e2v[i][m], logv[i][m] = e * e / v, np.log(2 * np.pi * v)
        if m + 1 < n_max:
            a = (gamma[m + 1] - phi @ gamma[m:0:-1]) / v
            phi = np.r_[phi - a * phi[::-1], a]
            v = v * (1 - a * a)
    return -0.5 * sum(a.sum() + b.sum() for a, b in zip(e2v, logv))

s = samples[0]
full_lags = np.arange(max(len(y) for y in data[s]) + 1)
check = []
for name in ["ARMA(1,1)", "ARMA(2,1)", "ARMA(2,2)"] + [f"GP {k}" for k in KERNELS] + ["GP two exponentials"]:
    x = fitted[s, name]
    if name.startswith("ARMA"):
        p, q = map(int, name[5:-1].split(","))
        g = arma_gamma(x, p, q, full_lags)
    elif name == "GP two exponentials":
        g = gp2_gamma(x, full_lags)
    else:
        g = gp_gamma(x, name[3:], full_lags)
    ll_k = res.loc[(res["sample"] == s) & (res["model"] == name), "LL"].item()
    check.append({"model": name, f"LL (K = {K})": ll_k, "LL exact": exact_loglik(data[s], g)})
check = pd.DataFrame(check).set_index("model")
check["difference"] = check.iloc[:, 0] - check.iloc[:, 1]
check.round(2)

## Step 14: which model wins?

ΔAIC and ΔBIC for every model (rows) and sample (columns). 0 = the best model for that sample; bigger = worse.

In [ ]:
dAIC = res.pivot(index="model", columns="sample", values="dAIC").loc[res["model"].unique()]
dAIC.round(0).astype(int)

In [ ]:
dBIC = res.pivot(index="model", columns="sample", values="dBIC").loc[res["model"].unique()]
dBIC.round(0).astype(int)

In [ ]:
# how often each model is the best, and the median rank across samples
pd.DataFrame({
    "best by AIC": dAIC.eq(0).sum(axis=1),
    "best by BIC": dBIC.eq(0).sum(axis=1),
    "median rank (AIC)": dAIC.rank().median(axis=1),
    "median rank (BIC)": dBIC.rank().median(axis=1),
}).sort_values("median rank (BIC)")

## Step 15: how many AR lags are needed?

AIC and BIC of AR(p) against p, relative to the best AR model of each sample.
If the curve is still falling at p = 20, depth depends on windows far back, which a model with a slow component (ARMA(2,1), two exponentials) can capture with fewer parameters.

In [ ]:
ar = res[res["model"].str.match(r"AR\(")].copy()
ar["p"] = ar["model"].str.extract(r"(\d+)").astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for crit, ax in zip(["AIC", "BIC"], axes):
    for s, g in ar.groupby("sample"):
        ax.plot(g["p"], g[crit] - g[crit].min(), marker="o", ms=3, lw=1, label=s)
    ax.set_yscale("symlog", linthresh=10)
    ax.set_xlabel("AR order p")
    ax.set_ylabel(f"{crit} − best AR {crit}")
axes[1].legend(fontsize=7, ncol=2)
fig.tight_layout()

## Step 16: fitted parameters across samples

- **AR(1):** φ and its correlation length −1/ln φ (in kb).
- **GP exponential:** length-scale ℓ, and the share of the variance that is nugget (independent window-to-window noise).
- **GP two exponentials:** the fast and slow length-scales, and the share of variance in each part.
  When one part gets almost no variance (share < 1%), the fit has collapsed to a single exponential and that part's length-scale means nothing, so it is shown as blank (NaN).

In [ ]:
def ar1_phi(s):
    return np.tanh(fitted[s, "AR(1)"][1])

params = []
for s in samples:
    phi1 = ar1_phi(s)
    s2, l, t2 = np.exp(fitted[s, "GP exponential"])
    v = np.exp(fitted[s, "GP two exponentials"])
    (sf, lf), (ss, ls) = sorted([(v[0], v[1]), (v[2], v[3])], key=lambda p: p[1])
    tot = sf + ss + v[4]
    params.append({
        "sample": s,
        "AR(1) phi": phi1,
        "AR(1) length (kb)": -WINDOW / 1e3 / np.log(phi1),
        "exp length (kb)": l * WINDOW / 1e3,
        "exp nugget share": t2 / (s2 + t2),
        "2exp fast length (kb)": lf * WINDOW / 1e3 if sf / tot > 0.01 else np.nan,
        "2exp slow length (kb)": ls * WINDOW / 1e3 if ss / tot > 0.01 else np.nan,
        "2exp fast share": sf / tot,
        "2exp slow share": ss / tot,
        "2exp nugget share": v[4] / tot,
    })
params = pd.DataFrame(params).set_index("sample")
params.round(3)

## Step 17: does the best model reproduce the real ACF?

A good LL is not the same as a realistic series. Simulate from each fitted model (same chromosome lengths as the real data),
compute its ACF exactly as in notebook 1, and lay it over the real ACF. A model that captures the process should follow the real curve, including the tail (log y-axis, right).

In [ ]:
def acf(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    var = (x ** 2).sum()
    return np.array([(x[:len(x) - k] * x[k:]).sum() / var for k in range(max_lag + 1)])

def mean_acf(ys):
    return np.mean([acf(y, MAX_LAG) for y in ys], axis=0)

def simulate(name, s, rng, burn=2000):
    lengths = [len(y) for y in data[s]]
    if name.startswith("HMM"):
        m = hmms[s, int(name.split()[1])]
        return [m.sample(n, random_state=rng.integers(1e9))[0][:, 0] for n in lengths]
    build = MODELS[name][0][0]
    phi, v = build(fitted[s, name])
    a = np.r_[1, -phi[K, 1:]]
    return [lfilter([1], a, rng.normal(0, np.sqrt(v[K]), n + burn))[burn:] for n in lengths]

SAMPLE = samples[0]
SHOW = ["iid", "AR(1)", "AR(5)", "ARMA(2,1)", "GP two exponentials", "GP Matern 3/2", "HMM 3 states"]

rng = np.random.default_rng(1)
lags_kb = np.arange(MAX_LAG + 1) * WINDOW / 1e3
real = mean_acf(data[SAMPLE])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, log in zip(axes, [False, True]):
    ax.plot(lags_kb[1:], real[1:], color="black", lw=2.5, label="real data")
    for name in SHOW:
        ax.plot(lags_kb[1:], mean_acf(simulate(name, SAMPLE, rng))[1:], lw=1, label=name)
    ax.set_xlabel("lag (kb)")
    ax.set_ylabel("autocorrelation")
    if log:
        ax.set_yscale("log")
        ax.set_ylim(5e-3, 1)
axes[0].set_title(f"{SAMPLE}: real ACF vs ACF of each fitted model")
axes[1].legend(fontsize=8)
fig.tight_layout()

Every sample, best model by BIC (simulated) against the real ACF:

In [ ]:
best_bic = dBIC.idxmin()
sims = {s: (mean_acf(simulate(best_bic[s], s, rng)), mean_acf(simulate("AR(1)", s, rng))) for s in samples}

n_col = 4
fig, axes = plt.subplots(int(np.ceil(len(samples) / n_col)), n_col, figsize=(14, 2.8 * np.ceil(len(samples) / n_col)), sharex=True, sharey=True)
for ax, s in zip(axes.flat, samples):
    ax.plot(lags_kb[1:], mean_acf(data[s])[1:], color="black", lw=2, label="real")
    ax.plot(lags_kb[1:], sims[s][0][1:], lw=1.2, label=best_bic[s])
    ax.plot(lags_kb[1:], sims[s][1][1:], lw=1, ls="--", label="AR(1)")
    ax.set_yscale("log")
    ax.set_ylim(5e-3, 1)
    ax.set_title(s, fontsize=9)
    ax.legend(fontsize=7)
for ax in axes.flat[len(samples):]:
    ax.set_visible(False)
fig.supxlabel("lag (kb)")
fig.supylabel("autocorrelation")
fig.tight_layout()

The same comparison as numbers: the ACF at a few lags for the real data, the best model by BIC, and AR(1).

In [ ]:
show_kb = [0.5, 2, 5, 10, 25, 50]
idx = [int(kb * 1e3 / WINDOW) for kb in show_kb]
table = {}
for s in samples:
    table[s, "real"] = mean_acf(data[s])[idx]
    table[s, best_bic[s]] = sims[s][0][idx]
    table[s, "AR(1)"] = sims[s][1][idx]
pd.DataFrame(table, index=pd.Index(show_kb, name="lag (kb)")).T.round(2)

## Things to keep in mind

- **Gaussian on the log scale.** All models except the HMM assume `y` is Gaussian. Real log-depth has heavier tails (and remaining zero windows at `log(PSEUDO)`), so the LLs are only approximate; the *ranking* of correlation structures is the useful part.
- **Stationary.** Every model assumes the same mean and correlation everywhere along the chromosome. Real depth also depends on local sequence (e.g. AT content, sWGA primer sites), which these models treat as part of the noise.
- **Huge n.** With ~44,000 windows, even small improvements in fit give large ΔAIC, so AIC tends to keep adding parameters. BIC is the more conservative guide here, and Step 17 (does it *look* right?) matters as much as the numbers.
- **Settings matter.** Try `TRIM_KB = 0` or a different `PSEUDO` and rerun from Step 3 to see whether the winner changes.